# NPS Alert Data Collection
Notebook goal: Initial repeat of previous notebook, then move to full data collection.



API documentation
https://www.nps.gov/subjects/developer/api-documentation.htm


## 1. Add the API key to Google Colab Secrets




In [ ]:
from google.colab import userdata

NPS_API_KEY = userdata.get("NPS_API_KEY")

if not NPS_API_KEY:
    raise ValueError(
        "NPS_API_KEY was not found. Add it to Google Colab Secrets "
        "and enable notebook access."
    )

print("API key loaded successfully.")


API key loaded successfully.


## 2. Import packages and define the alerts endpoint

The first request will retrieve only five alerts so that the connection and response format can be checked before collecting more data.


In [ ]:
import json
import requests
import pandas as pd

BASE_URL = "https://developer.nps.gov/api/v1/alerts"

headers = {
    "X-Api-Key": NPS_API_KEY
}

params = {
    "limit": 5,
    "start": 0
}

print("Packages imported and request settings created.")


Packages imported and request settings created.


## 3. Make the initial API request




In [ ]:
try:
    response = requests.get(
        BASE_URL,
        headers=headers,
        params=params,
        timeout=30
    )

    print("Status code:", response.status_code)
    print("Request URL:", response.url)

    response.raise_for_status()

except requests.exceptions.Timeout as exc:
    raise RuntimeError("The NPS API request timed out.") from exc

except requests.exceptions.HTTPError as exc:
    raise RuntimeError(
        f"The NPS API returned an HTTP error: {response.status_code}\n"
        f"Response preview: {response.text[:500]}"
    ) from exc

except requests.exceptions.RequestException as exc:
    raise RuntimeError(f"The NPS API request failed: {exc}") from exc


Status code: 200
Request URL: https://developer.nps.gov/api/v1/alerts?limit=5&start=0


## 4. Inspect the top-level response

This step checks the overall response structure and the number of alerts reported by the API.


In [ ]:
payload = response.json()

print("Top-level keys:", list(payload.keys()))
print("Reported total alerts:", payload.get("total"))
print("Number returned in this request:", len(payload.get("data", [])))


Top-level keys: ['total', 'limit', 'start', 'data']
Reported total alerts: 625
Number returned in this request: 5


## 5. Inspect the fields in the first alert

The project should use the fields that the API actually returns rather than assuming that every proposed field is available.


In [ ]:
alerts = payload.get("data", [])

if alerts:
    first_alert = alerts[0]

    print("Fields in one alert:")
    for field in first_alert.keys():
        print("-", field)
else:
    print("The request succeeded, but no alert records were returned.")


Fields in one alert:
- id
- url
- title
- parkCode
- description
- category
- relatedRoadEvents
- lastIndexedDate


## 6. View the complete first alert



In [ ]:
if alerts:
    print(json.dumps(alerts[0], indent=2))
else:
    print("No alert is available to display.")


{
  "id": "9C29DF02-C5E8-4008-973F-26C5B8A7FC52",
  "url": "",
  "title": "Tower of Voices Maintenance",
  "parkCode": "flni",
  "description": "The Tower of Voices is currently open and is undergoing windchime testing. Currently four (4) windchimes have been removed from the Tower for maintenance and will be replaced as soon as possible.",
  "category": "Information",
  "relatedRoadEvents": [],
  "lastIndexedDate": "2026-06-22 00:00:00.0"
}


## 7. Convert the sample to a pandas DataFrame


In [ ]:
sample_df = pd.DataFrame(alerts)

print("Shape:", sample_df.shape)
display(sample_df.head())


Shape: (5, 8)


,id,url,title,parkCode,description,category,relatedRoadEvents,lastIndexedDate
0,9C29DF02-C5E8-4008-973F-26C5B8A7FC52,,Tower of Voices Maintenance,flni,The Tower of Voices is currently open and is u...,Information,[],2026-06-22 00:00:00.0
1,17F8CB55-3003-412B-97CB-13533AC195F4,,Andrew Johnson NHS Visitor Center Closed for T...,anjo,Andrew Johnson Visitor Center will be closed M...,Park Closure,[],2026-06-22 00:00:00.0
2,2F5C3616-F71A-4358-8394-DA1A758E2F00,,River Road Closure - June 22-24,dewa,River Road will be closed between Park Headqua...,Park Closure,"[{'title': 'River Road', 'id': '2F5C3616-F71A-...",2026-06-22 00:00:00.0
3,97783AC4-B242-4BF7-8C71-37A17B987928,https://www.nps.gov/glac/planyourvisit/index.htm,Going-to-the-Sun Road is Open for 2026 Season,glac,The Going-to-the-Sun Road is fully open for th...,Information,[],2026-06-22 00:00:00.0
4,D547F69E-F5AD-4F7B-B6A5-D838C3BC05C7,,Volcano Road Closed June 22 8:30am - 4:30pm,cavo,"Due to road repair and maintenance, volcano ro...",Park Closure,[],2026-06-22 00:00:00.0


## 8. List the available columns


In [ ]:
print(sample_df.columns.tolist())


['id', 'url', 'title', 'parkCode', 'description', 'category', 'relatedRoadEvents', 'lastIndexedDate']


## 9. Display candidate project fields

This cell selects only candidate fields that are actually present, so it will not fail if the API omits one of the proposed fields.


In [ ]:
candidate_columns = [
    "id",
    "parkCode",
    "title",
    "description",
    "category",
    "url",
    "lastIndexedDate"
]

available_columns = [
    column
    for column in candidate_columns
    if column in sample_df.columns
]

print("Candidate fields available:", available_columns)

if available_columns:
    display(sample_df[available_columns])
else:
    print("None of the candidate columns were found.")


Candidate fields available: ['id', 'parkCode', 'title', 'description', 'category', 'url', 'lastIndexedDate']


,id,parkCode,title,description,category,url,lastIndexedDate
0,9C29DF02-C5E8-4008-973F-26C5B8A7FC52,flni,Tower of Voices Maintenance,The Tower of Voices is currently open and is u...,Information,,2026-06-22 00:00:00.0
1,17F8CB55-3003-412B-97CB-13533AC195F4,anjo,Andrew Johnson NHS Visitor Center Closed for T...,Andrew Johnson Visitor Center will be closed M...,Park Closure,,2026-06-22 00:00:00.0
2,2F5C3616-F71A-4358-8394-DA1A758E2F00,dewa,River Road Closure - June 22-24,River Road will be closed between Park Headqua...,Park Closure,,2026-06-22 00:00:00.0
3,97783AC4-B242-4BF7-8C71-37A17B987928,glac,Going-to-the-Sun Road is Open for 2026 Season,The Going-to-the-Sun Road is fully open for th...,Information,https://www.nps.gov/glac/planyourvisit/index.htm,2026-06-22 00:00:00.0
4,D547F69E-F5AD-4F7B-B6A5-D838C3BC05C7,cavo,Volcano Road Closed June 22 8:30am - 4:30pm,"Due to road repair and maintenance, volcano ro...",Park Closure,,2026-06-22 00:00:00.0


## 10. Read the first five alerts in a traveler-friendly format



In [ ]:
if not alerts:
    print("No alerts were returned.")

for index, alert in enumerate(alerts, start=1):
    print("=" * 80)
    print(f"ALERT {index}")
    print("Park code:", alert.get("parkCode"))
    print("NPS category:", alert.get("category"))
    print("Title:", alert.get("title"))
    print("Description:", alert.get("description"))
    print("URL:", alert.get("url"))


ALERT 1
Park code: flni
NPS category: Information
Title: Tower of Voices Maintenance
Description: The Tower of Voices is currently open and is undergoing windchime testing. Currently four (4) windchimes have been removed from the Tower for maintenance and will be replaced as soon as possible.
URL: 
ALERT 2
Park code: anjo
NPS category: Park Closure
Title: Andrew Johnson NHS Visitor Center Closed for Two Days (June 22/23).
Description: Andrew Johnson Visitor Center will be closed Monday/Tuesday, June 22/23, 2026. A Visitor Contact Station will be operational for both days at the Early Home facility (201 East Depot Street), adjacent to the main parking lot.
URL: 
ALERT 3
Park code: dewa
NPS category: Park Closure
Title: River Road Closure - June 22-24
Description: River Road will be closed between Park Headquarters and Smithfield Beach from 8 a.m. to 3:30 p.m. on June 22 - 24 for shoulder work, culvert cleaning, and other roadway maintenance. Please plan accordingly and use alternate rou

# Collect and Inspect the Full Active-Alert Dataset

The five-record API test was successful. Move to full dataset.

## 12. Define a reusable paginated API function

The function below repeatedly requests batches until it reaches the total number of records reported by the API.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import time

def collect_paginated_nps_data(
    endpoint,
    headers,
    extra_params=None,
    batch_size=50,
    pause_seconds=0.2
):
    """Collect all records from a paginated NPS API endpoint."""

    all_records = []
    start = 0
    reported_total = None
    extra_params = extra_params or {}

    while True:
        request_params = {
            **extra_params,
            "limit": batch_size,
            "start": start
        }

        response = requests.get(
            endpoint,
            headers=headers,
            params=request_params,
            timeout=30
        )
        response.raise_for_status()

        page = response.json()
        records = page.get("data", [])

        if reported_total is None:
            reported_total = int(page.get("total", 0))
            print(f"API reports {reported_total:,} total records.")

        all_records.extend(records)
        print(
            f"Collected {len(all_records):,} of "
            f"{reported_total:,} records."
        )

        if not records or len(all_records) >= reported_total:
            break

        start += len(records)
        time.sleep(pause_seconds)

    return all_records, reported_total


## 13. Collect all active alerts


In [ ]:
ALERTS_ENDPOINT = "https://developer.nps.gov/api/v1/alerts"

all_alerts, reported_alert_total = collect_paginated_nps_data(
    endpoint=ALERTS_ENDPOINT,
    headers=headers,
    batch_size=50
)

print("\nCollection complete.")
print("Reported total:", reported_alert_total)
print("Records collected:", len(all_alerts))


API reports 625 total records.
Collected 50 of 625 records.
Collected 100 of 625 records.
Collected 150 of 625 records.
Collected 200 of 625 records.
Collected 250 of 625 records.
Collected 300 of 625 records.
Collected 350 of 625 records.
Collected 400 of 625 records.
Collected 450 of 625 records.
Collected 500 of 625 records.
Collected 550 of 625 records.
Collected 600 of 625 records.
Collected 625 of 625 records.

Collection complete.
Reported total: 625
Records collected: 625


## 14. Create a collection timestamp and output folder


In [ ]:
collection_time_utc = datetime.now(timezone.utc)
collection_timestamp = collection_time_utc.isoformat()
collection_date = collection_time_utc.strftime("%Y-%m-%d")
timestamp_for_filename = collection_time_utc.strftime("%Y%m%dT%H%M%SZ")

output_dir = Path("nps_alert_data")
output_dir.mkdir(exist_ok=True)

print("Collection timestamp:", collection_timestamp)
print("Output folder:", output_dir.resolve())


Collection timestamp: 2026-06-22T17:51:37.242182+00:00
Output folder: /content/nps_alert_data


## 15. Save the raw API response as JSON

This preserves the original API records before cleaning or transformation.


In [ ]:
raw_json_path = output_dir / f"nps_alerts_raw_{timestamp_for_filename}.json"

raw_snapshot = {
    "collection_timestamp_utc": collection_timestamp,
    "reported_total": reported_alert_total,
    "records_collected": len(all_alerts),
    "data": all_alerts
}

with raw_json_path.open("w", encoding="utf-8") as file:
    json.dump(raw_snapshot, file, indent=2, ensure_ascii=False)

print("Saved raw JSON:", raw_json_path)


Saved raw JSON: nps_alert_data/nps_alerts_raw_20260622T175137Z.json


## 16. Convert the complete alert collection to a DataFrame


In [ ]:
alerts_df = pd.DataFrame(all_alerts)

alerts_df["collection_timestamp_utc"] = collection_timestamp
alerts_df["collection_date"] = collection_date

print("Alert DataFrame shape:", alerts_df.shape)
print("\nColumns:")
print(alerts_df.columns.tolist())

display(alerts_df.head())


Alert DataFrame shape: (625, 10)

Columns:
['id', 'url', 'title', 'parkCode', 'description', 'category', 'relatedRoadEvents', 'lastIndexedDate', 'collection_timestamp_utc', 'collection_date']


,id,url,title,parkCode,description,category,relatedRoadEvents,lastIndexedDate,collection_timestamp_utc,collection_date
0,9C29DF02-C5E8-4008-973F-26C5B8A7FC52,,Tower of Voices Maintenance,flni,The Tower of Voices is currently open and is u...,Information,[],2026-06-22 00:00:00.0,2026-06-22T17:51:37.242182+00:00,2026-06-22
1,17F8CB55-3003-412B-97CB-13533AC195F4,,Andrew Johnson NHS Visitor Center Closed for T...,anjo,Andrew Johnson Visitor Center will be closed M...,Park Closure,[],2026-06-22 00:00:00.0,2026-06-22T17:51:37.242182+00:00,2026-06-22
2,2F5C3616-F71A-4358-8394-DA1A758E2F00,,River Road Closure - June 22-24,dewa,River Road will be closed between Park Headqua...,Park Closure,"[{'title': 'River Road', 'id': '2F5C3616-F71A-...",2026-06-22 00:00:00.0,2026-06-22T17:51:37.242182+00:00,2026-06-22
3,97783AC4-B242-4BF7-8C71-37A17B987928,https://www.nps.gov/glac/planyourvisit/index.htm,Going-to-the-Sun Road is Open for 2026 Season,glac,The Going-to-the-Sun Road is fully open for th...,Information,[],2026-06-22 00:00:00.0,2026-06-22T17:51:37.242182+00:00,2026-06-22
4,D547F69E-F5AD-4F7B-B6A5-D838C3BC05C7,,Volcano Road Closed June 22 8:30am - 4:30pm,cavo,"Due to road repair and maintenance, volcano ro...",Park Closure,[],2026-06-22 00:00:00.0,2026-06-22T17:51:37.242182+00:00,2026-06-22


## 17. Save the initial structured CSV

This file keeps the API fields as returned, with only the collection timestamp added.


In [ ]:
initial_csv_path = (
    output_dir / f"nps_alerts_structured_{timestamp_for_filename}.csv"
)

alerts_df.to_csv(initial_csv_path, index=False)

print("Saved structured CSV:", initial_csv_path)


Saved structured CSV: nps_alert_data/nps_alerts_structured_20260622T175137Z.csv


## 18. Verify totals and unique alert IDs


In [ ]:
print("API-reported total:", reported_alert_total)
print("Rows collected:", len(alerts_df))

if "id" in alerts_df.columns:
    print("Unique alert IDs:", alerts_df["id"].nunique())
    print("Duplicate alert-ID rows:", alerts_df.duplicated(subset="id").sum())
else:
    print("No alert ID field was found.")


API-reported total: 625
Rows collected: 625
Unique alert IDs: 625
Duplicate alert-ID rows: 0


## 19. Inspect missing values in important fields


In [ ]:
important_fields = [
    "id",
    "parkCode",
    "title",
    "description",
    "category",
    "url",
    "lastIndexedDate"
]

available_important_fields = [
    field for field in important_fields
    if field in alerts_df.columns
]

missing_summary = pd.DataFrame({
    "field": available_important_fields,
    "missing_count": [
        alerts_df[field].isna().sum()
        + alerts_df[field].astype(str).str.strip().eq("").sum()
        for field in available_important_fields
    ]
})

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / len(alerts_df) * 100
).round(2)

display(missing_summary)


,field,missing_count,missing_percent
0,id,0,0.00
1,parkCode,1,0.16
2,title,0,0.00
3,description,0,0.00
4,category,0,0.00
5,url,343,54.88
6,lastIndexedDate,0,0.00


## 20. Inspect the original NPS alert-category distribution


In [ ]:
if "category" in alerts_df.columns:
    category_counts = (
        alerts_df["category"]
        .fillna("Missing")
        .replace("", "Missing")
        .value_counts()
        .rename_axis("nps_category")
        .reset_index(name="alert_count")
    )

    category_counts["percent"] = (
        category_counts["alert_count"] / len(alerts_df) * 100
    ).round(2)

    display(category_counts)
else:
    print("The category field is unavailable.")


,nps_category,alert_count,percent
0,Information,327,52.32
1,Park Closure,157,25.12
2,Caution,129,20.64
3,Danger,12,1.92


## 21. Measure alert-text length

The combined text will later support classification. These summaries help determine whether alerts are generally long enough to classify and whether a maximum transformer sequence length is needed.


In [ ]:
title_text = (
    alerts_df["title"].fillna("").astype(str)
    if "title" in alerts_df.columns
    else pd.Series("", index=alerts_df.index)
)

description_text = (
    alerts_df["description"].fillna("").astype(str)
    if "description" in alerts_df.columns
    else pd.Series("", index=alerts_df.index)
)

alerts_df["combined_text"] = (
    title_text.str.strip()
    + " "
    + description_text.str.strip()
).str.strip()

alerts_df["title_word_count"] = title_text.str.split().str.len()
alerts_df["description_word_count"] = description_text.str.split().str.len()
alerts_df["combined_word_count"] = alerts_df["combined_text"].str.split().str.len()
alerts_df["combined_character_count"] = alerts_df["combined_text"].str.len()

display(
    alerts_df[
        [
            "title_word_count",
            "description_word_count",
            "combined_word_count",
            "combined_character_count"
        ]
    ].describe(percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99])
)

,title_word_count,description_word_count,combined_word_count,combined_character_count
count,625.000000,625.000000,625.000000,625.000000
mean,6.496000,37.771200,44.267200,273.528000
std,3.079669,15.692982,16.412861,101.423804
min,1.000000,2.000000,4.000000,21.000000
25%,4.000000,27.000000,33.000000,206.000000
50%,6.000000,37.000000,43.000000,268.000000
75%,8.000000,44.000000,52.000000,312.000000
90%,11.000000,59.600000,67.000000,416.600000
95%,12.000000,71.800000,78.800000,497.000000
99%,14.000000,81.000000,89.760000,552.760000


## 22. Identify empty and very short records


In [ ]:
empty_text_count = alerts_df["combined_text"].str.strip().eq("").sum()
very_short_count = alerts_df["combined_word_count"].lt(10).sum()

print("Empty combined-text records:", empty_text_count)
print("Records with fewer than 10 words:", very_short_count)

short_alert_columns = [
    column for column in
    ["id", "parkCode", "category", "title", "description", "combined_word_count"]
    if column in alerts_df.columns
]

display(
    alerts_df.loc[
        alerts_df["combined_word_count"].lt(10),
        short_alert_columns
    ].sort_values("combined_word_count").head(20)
)


Empty combined-text records: 0
Records with fewer than 10 words: 2


,id,parkCode,category,title,description,combined_word_count
619,029622AA-F8C4-40E2-99C1-13479E55D4ED,inau,Danger,Test Alert,Test Alert,4
243,2F801986-6885-4AD0-B868-1A20A47DAD80,moja,Park Closure,Hart Mine Road Closed,Hart Mine Road Closed.,8


## 23. Collect park names from the NPS parks endpoint

The alert response supplies park codes, but the full park names are helpful for interpretation and held-out-park evaluation.


In [ ]:
PARKS_ENDPOINT = "https://developer.nps.gov/api/v1/parks"

all_parks, reported_park_total = collect_paginated_nps_data(
    endpoint=PARKS_ENDPOINT,
    headers=headers,
    batch_size=50
)

parks_df = pd.DataFrame(all_parks)

print("\nPark DataFrame shape:", parks_df.shape)
print("Park columns:", parks_df.columns.tolist())

park_lookup_columns = [
    column for column in
    ["parkCode", "fullName", "name", "states", "designation"]
    if column in parks_df.columns
]

park_lookup_df = (
    parks_df[park_lookup_columns]
    .drop_duplicates(subset="parkCode")
    .copy()
)

display(park_lookup_df.head())


API reports 474 total records.
Collected 50 of 474 records.
Collected 100 of 474 records.
Collected 150 of 474 records.
Collected 200 of 474 records.
Collected 250 of 474 records.
Collected 300 of 474 records.
Collected 350 of 474 records.
Collected 400 of 474 records.
Collected 450 of 474 records.
Collected 474 of 474 records.

Park DataFrame shape: (474, 25)
Park columns: ['id', 'url', 'fullName', 'parkCode', 'description', 'latitude', 'longitude', 'latLong', 'activities', 'topics', 'states', 'contacts', 'entranceFees', 'entrancePasses', 'fees', 'directionsInfo', 'directionsUrl', 'operatingHours', 'addresses', 'images', 'weatherInfo', 'name', 'designation', 'multimedia', 'relevanceScore']


,parkCode,fullName,name,states,designation
0,abli,Abraham Lincoln Birthplace National Historical...,Abraham Lincoln Birthplace,KY,National Historical Park
1,acad,Acadia National Park,Acadia,ME,National Park
2,adam,Adams National Historical Park,Adams,MA,National Historical Park
3,afam,African American Civil War Memorial,African American Civil War Memorial,DC,
4,afbg,African Burial Ground National Monument,African Burial Ground,NY,National Monument


## 24. Join full park information to the alerts


In [ ]:
alerts_enriched_df = alerts_df.merge(
    park_lookup_df,
    on="parkCode",
    how="left",
    validate="many_to_one"
)

if "fullName" in alerts_enriched_df.columns:
    print(
        "Alerts without a matched full park name:",
        alerts_enriched_df["fullName"].isna().sum()
    )

print("Enriched DataFrame shape:", alerts_enriched_df.shape)
display(alerts_enriched_df.head())


Alerts without a matched full park name: 4
Enriched DataFrame shape: (625, 19)


,id,url,title,parkCode,description,category,relatedRoadEvents,lastIndexedDate,collection_timestamp_utc,collection_date,combined_text,title_word_count,description_word_count,combined_word_count,combined_character_count,fullName,name,states,designation
0,9C29DF02-C5E8-4008-973F-26C5B8A7FC52,,Tower of Voices Maintenance,flni,The Tower of Voices is currently open and is u...,Information,[],2026-06-22 00:00:00.0,2026-06-22T17:51:37.242182+00:00,2026-06-22,Tower of Voices Maintenance The Tower of Voice...,4,32,36,223,Flight 93 National Memorial,Flight 93,PA,National Memorial
1,17F8CB55-3003-412B-97CB-13533AC195F4,,Andrew Johnson NHS Visitor Center Closed for T...,anjo,Andrew Johnson Visitor Center will be closed M...,Park Closure,[],2026-06-22 00:00:00.0,2026-06-22T17:51:37.242182+00:00,2026-06-22,Andrew Johnson NHS Visitor Center Closed for T...,11,36,47,292,Andrew Johnson National Historic Site,Andrew Johnson,TN,National Historic Site
2,2F5C3616-F71A-4358-8394-DA1A758E2F00,,River Road Closure - June 22-24,dewa,River Road will be closed between Park Headqua...,Park Closure,"[{'title': 'River Road', 'id': '2F5C3616-F71A-...",2026-06-22 00:00:00.0,2026-06-22T17:51:37.242182+00:00,2026-06-22,River Road Closure - June 22-24 River Road wil...,6,38,44,263,Delaware Water Gap National Recreation Area,Delaware Water Gap,"NJ,PA",National Recreation Area
3,97783AC4-B242-4BF7-8C71-37A17B987928,https://www.nps.gov/glac/planyourvisit/index.htm,Going-to-the-Sun Road is Open for 2026 Season,glac,The Going-to-the-Sun Road is fully open for th...,Information,[],2026-06-22 00:00:00.0,2026-06-22T17:51:37.242182+00:00,2026-06-22,Going-to-the-Sun Road is Open for 2026 Season ...,7,32,39,231,Glacier National Park,Glacier,MT,National Park
4,D547F69E-F5AD-4F7B-B6A5-D838C3BC05C7,,Volcano Road Closed June 22 8:30am - 4:30pm,cavo,"Due to road repair and maintenance, volcano ro...",Park Closure,[],2026-06-22 00:00:00.0,2026-06-22T17:51:37.242182+00:00,2026-06-22,Volcano Road Closed June 22 8:30am - 4:30pm Du...,8,31,39,226,Capulin Volcano National Monument,Capulin Volcano,NM,National Monument


## 25. Inspect the distribution of alerts across parks


In [ ]:
park_name_column = (
    "fullName"
    if "fullName" in alerts_enriched_df.columns
    else "parkCode"
)

alerts_by_park = (
    alerts_enriched_df
    .groupby(["parkCode", park_name_column], dropna=False)
    .size()
    .reset_index(name="alert_count")
    .sort_values("alert_count", ascending=False)
)

print("Unique park codes represented:", alerts_enriched_df["parkCode"].nunique())
display(alerts_by_park.head(25))


Unique park codes represented: 289


,parkCode,fullName,alert_count
144,iatr,Ice Age National Scenic Trail,10
199,noca,North Cascades National Park,9
121,goga,Golden Gate National Recreation Area,8
184,moja,Mojave National Preserve,7
279,whsa,White Sands National Park,6
112,gate,Gateway National Recreation Area,6
169,lavo,Lassen Volcanic National Park,6
22,beol,Bent's Old Fort National Historic Site,6
268,vafo,Valley Forge National Historical Park,5
210,pagr,Paterson Great Falls National Historical Park,5


## 26. Check exact duplicate text

Duplicate text is not automatically removed here. First, identify duplicate groups and inspect whether they represent repeated records, common templates, or legitimate park-specific alerts.


In [ ]:
normalized_text = (
    alerts_enriched_df["combined_text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

alerts_enriched_df["normalized_text"] = normalized_text

exact_duplicate_mask = alerts_enriched_df.duplicated(
    subset="normalized_text",
    keep=False
) & alerts_enriched_df["normalized_text"].ne("")

exact_duplicate_rows = alerts_enriched_df.loc[
    exact_duplicate_mask,
    [
        column for column in
        ["id", "parkCode", park_name_column, "category", "title", "normalized_text"]
        if column in alerts_enriched_df.columns
    ]
].sort_values("normalized_text")

print("Rows belonging to exact duplicate-text groups:", len(exact_duplicate_rows))
print(
    "Unique duplicated text values:",
    exact_duplicate_rows["normalized_text"].nunique()
    if not exact_duplicate_rows.empty else 0
)

display(exact_duplicate_rows.head(30))


Rows belonging to exact duplicate-text groups: 38
Unique duplicated text values: 13


,id,parkCode,fullName,category,title,normalized_text
429,B831CEB1-40B7-455C-9E09-644D8291FC7A,goga,Golden Gate National Recreation Area,Caution,Auto-Theft Warning - Only Bring What You Need ...,auto-theft warning - only bring what you need ...
551,B127187F-76DD-4147-8AB5-CEC00AA6700D,muwo,Muir Woods National Monument,Caution,Auto-Theft Warning - Only Bring What You Need ...,auto-theft warning - only bring what you need ...
552,FDC95683-8853-4829-8A84-6E7E886B1C7A,prsf,Presidio of San Francisco,Caution,Auto-Theft Warning - Only Bring What You Need ...,auto-theft warning - only bring what you need ...
553,B7A06513-5CF2-4CC8-9091-3B80E52C2D0B,fopo,Fort Point National Historic Site,Caution,Auto-Theft Warning - Only Bring What You Need ...,auto-theft warning - only bring what you need ...
481,800F736B-76E9-46C8-AFD9-B756F76EBC32,hamp,Hampton National Historic Site,Information,Baltimore Area Traffic Advisory,baltimore area traffic advisory traffic in the...
483,990FA2AA-0B32-4DA1-9584-7C6DA8CF4EF4,fomc,Fort McHenry National Monument and Historic Sh...,Information,Baltimore Area Traffic Advisory,baltimore area traffic advisory traffic in the...
11,BCB1ABBF-4FA2-4334-916D-4D206017C113,fopo,Fort Point National Historic Site,Caution,Beach Hazards Statement in Effect until 5 PM W...,beach hazards statement in effect until 5 pm w...
13,8A1F9B2A-17EC-4824-80D7-DFE1616DC4FA,prsf,Presidio of San Francisco,Caution,Beach Hazards Statement in Effect until 5 PM W...,beach hazards statement in effect until 5 pm w...
14,7A4A704E-7103-48EF-87F5-45D3BB2D733E,goga,Golden Gate National Recreation Area,Caution,Beach Hazards Statement in Effect until 5 PM W...,beach hazards statement in effect until 5 pm w...
82,6E732798-DC47-424B-8FA8-2FC6E365D3F7,camo,Castle Mountains National Monument,Caution,Fire restrictions in place,fire restrictions in place fire restrictions a...


## 27. Save the enriched working dataset


In [ ]:
enriched_csv_path = (
    output_dir / f"nps_alerts_enriched_{timestamp_for_filename}.csv"
)

alerts_enriched_df.to_csv(enriched_csv_path, index=False)

print("Saved enriched CSV:", enriched_csv_path)


Saved enriched CSV: nps_alert_data/nps_alerts_enriched_20260622T175137Z.csv


## 28. Full-Collection Feasibility Check


625 unique alerts
289 park codes
No missing titles or descriptions
Only 2 alerts under 10 words
Average combined length of about 44 words
13 exact duplicate-text groups, involving 38 records
No single park dominates the dataset; the largest park has only 10 alerts

The original NPS categories are uneven:

Information: 327
Park Closure: 157
Caution: 129
Danger: 12

This means a random taxonomy-review sample would contain few Danger alerts. It also confirms that the NPS categories should remain metadata rather than become traveler-centered target labels.

A few records need special treatment:

One obvious “Test Alert” should be excluded.
Four alerts did not match a full park name.
One record is missing a park code.
Exact duplicate texts should not be allowed to appear across later training and test sets.

